# 🏭 Business Optimization Model using Linear Programming
## Task 4 — End-to-End Optimization with PuLP

**Author:** Kartikay Verma  
**Problem:** Multi-Product, Multi-Machine **Production Planning Optimization**  
**Technique:** Linear Programming (LP) + Integer Linear Programming (ILP)  
**Library:** PuLP (Python LP Solver)

---

### 📌 Business Context

> **TechManufacture Pvt. Ltd.** produces **5 electronics products** (Laptop, Tablet, Smartphone, Smartwatch, Headphones) on **3 machines** (Assembly, Testing, Packaging). Each machine has a fixed weekly capacity, raw materials are limited, and the company has a weekly production budget.
>
> **Goal:** Find the optimal **number of units to produce per product per week** that **maximises total profit** while respecting all constraints.

### 📋 Models Built

| # | Model | Purpose |
|---|-------|---------|
| 1 | LP (Continuous) | Maximise profit — fractional units allowed |
| 2 | ILP (Integer)   | Same but units must be whole numbers |
| 3 | Sensitivity     | How does profit change as budget increases? |
| 4 | What-If         | Impact of semiconductor supply shock (−20%) |

---
## 1. Library Imports

In [ ]:
# Install if running for the first time
# !pip install pulp matplotlib seaborn pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pulp

print(f"PuLP version : {pulp.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version : {np.__version__}")
print("\n✅ All libraries loaded successfully!")

---
## 2. Business Data — Problem Definition

We define all data as Python dictionaries. This makes the model easy to update as the business changes.

In [ ]:
# ─── Products ───
PRODUCTS = ["Laptop", "Tablet", "Smartphone", "Smartwatch", "Headphones"]

# Profit per unit (₹)
PROFIT = {
    "Laptop":     15_000,
    "Tablet":      8_000,
    "Smartphone": 12_000,
    "Smartwatch":  5_500,
    "Headphones":  3_200,
}

# Max market demand per week (units)
DEMAND = {
    "Laptop":     200,
    "Tablet":     350,
    "Smartphone": 500,
    "Smartwatch": 400,
    "Headphones": 600,
}

# Minimum contractual production per week (units)
MIN_PROD = {
    "Laptop":      20,
    "Tablet":      30,
    "Smartphone":  50,
    "Smartwatch":  25,
    "Headphones":  40,
}

# Variable cost per unit (₹)
VARIABLE_COST = {
    "Laptop":      6_000,
    "Tablet":      3_200,
    "Smartphone":  4_500,
    "Smartwatch":  2_000,
    "Headphones":    800,
}

# Display as table
product_df = pd.DataFrame({
    "Product":       PRODUCTS,
    "Profit/unit (₹)": [PROFIT[p]        for p in PRODUCTS],
    "Cost/unit (₹)":   [VARIABLE_COST[p] for p in PRODUCTS],
    "Min Demand":      [MIN_PROD[p]       for p in PRODUCTS],
    "Max Demand":      [DEMAND[p]         for p in PRODUCTS],
}).set_index("Product")

print("📦 Product Data:")
product_df

In [ ]:
# ─── Machine Constraints ───
MACHINES = ["Assembly", "Testing", "Packaging"]

MACHINE_HOURS = {
    "Laptop":     {"Assembly": 4.5, "Testing": 2.0, "Packaging": 0.8},
    "Tablet":     {"Assembly": 2.0, "Testing": 1.5, "Packaging": 0.5},
    "Smartphone": {"Assembly": 1.8, "Testing": 1.2, "Packaging": 0.4},
    "Smartwatch": {"Assembly": 1.0, "Testing": 0.8, "Packaging": 0.3},
    "Headphones": {"Assembly": 0.6, "Testing": 0.4, "Packaging": 0.2},
}

MACHINE_CAPACITY = {
    "Assembly":  1_800,   # hours/week
    "Testing":   1_200,
    "Packaging":   600,
}

machine_hours_df = pd.DataFrame(MACHINE_HOURS).T
machine_hours_df["(hours/unit) →"] = ""
print("🔧 Machine Hours Required per Unit (hours):")
print(machine_hours_df.drop(columns="(hours/unit) →"))
print(f"\n⏱️  Machine Capacities (hours/week): {MACHINE_CAPACITY}")

In [ ]:
# ─── Material & Budget Constraints ───
MATERIALS = ["Semiconductors", "Plastics", "Metals"]

MATERIAL_USE = {
    "Laptop":     {"Semiconductors": 0.5, "Plastics": 1.2, "Metals": 0.8},
    "Tablet":     {"Semiconductors": 0.3, "Plastics": 0.8, "Metals": 0.3},
    "Smartphone": {"Semiconductors": 0.4, "Plastics": 0.5, "Metals": 0.2},
    "Smartwatch": {"Semiconductors": 0.2, "Plastics": 0.2, "Metals": 0.1},
    "Headphones": {"Semiconductors": 0.1, "Plastics": 0.3, "Metals": 0.1},
}

MATERIAL_SUPPLY = {
    "Semiconductors": 300,   # kg/week
    "Plastics":       600,
    "Metals":         400,
}

WEEKLY_BUDGET = 5_000_000   # ₹50 lakh/week

print("🧪 Material Use per Unit (kg):")
print(pd.DataFrame(MATERIAL_USE).T)
print(f"\n📦 Material Supply (kg/week): {MATERIAL_SUPPLY}")
print(f"💰 Weekly Budget: ₹{WEEKLY_BUDGET:,}")

---
## 3. Mathematical Formulation

### Decision Variables
$$x_i = \text{units of product } i \text{ to produce per week} \quad \forall i \in \{\text{Laptop, Tablet, Smartphone, Smartwatch, Headphones}\}$$

### Objective Function — Maximise Total Profit
$$\text{Maximise } Z = \sum_{i} \text{profit}_i \cdot x_i$$

### Constraints

| Constraint Type | Formula |
|---|---|
| **Demand upper bound** | $x_i \le \text{demand}_i$ |
| **Minimum production** | $x_i \ge \text{min\_prod}_i$ |
| **Machine capacity**   | $\sum_i \text{hours}_{im} \cdot x_i \le \text{capacity}_m \quad \forall m$ |
| **Material supply**    | $\sum_i \text{material}_{ij} \cdot x_i \le \text{supply}_j \quad \forall j$ |
| **Budget**             | $\sum_i \text{cost}_i \cdot x_i \le \text{budget}$ |
| **Non-negativity**     | $x_i \ge 0$ |

---
## 4. Model 1 — Linear Programming (LP)

Fractional units allowed; gives an upper bound on achievable profit.

In [ ]:
# ──────────────────────────────────────────────────────────────
#  BUILD LP MODEL
# ──────────────────────────────────────────────────────────────
prob_lp = pulp.LpProblem("Production_Planning_LP", pulp.LpMaximize)

# Decision variables (continuous)
x = {p: pulp.LpVariable(f"x_{p}", lowBound=0, cat="Continuous") for p in PRODUCTS}

# ── Objective ──
prob_lp += pulp.lpSum(PROFIT[p] * x[p] for p in PRODUCTS), "Total_Profit"

# ── Constraints ──
# 1. Demand upper bounds
for p in PRODUCTS:
    prob_lp += x[p] <= DEMAND[p],   f"Demand_UB_{p}"

# 2. Minimum contractual production
for p in PRODUCTS:
    prob_lp += x[p] >= MIN_PROD[p], f"MinProd_{p}"

# 3. Machine hour constraints
for m in MACHINES:
    prob_lp += (
        pulp.lpSum(MACHINE_HOURS[p][m] * x[p] for p in PRODUCTS) <= MACHINE_CAPACITY[m],
        f"Machine_{m}"
    )

# 4. Material supply constraints
for mat in MATERIALS:
    prob_lp += (
        pulp.lpSum(MATERIAL_USE[p][mat] * x[p] for p in PRODUCTS) <= MATERIAL_SUPPLY[mat],
        f"Material_{mat}"
    )

# 5. Budget constraint
prob_lp += (
    pulp.lpSum(VARIABLE_COST[p] * x[p] for p in PRODUCTS) <= WEEKLY_BUDGET,
    "Budget"
)

print("LP Model structure:")
print(prob_lp)

In [ ]:
# ── SOLVE ──
prob_lp.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Solver Status : {pulp.LpStatus[prob_lp.status]}")
print(f"Optimal Profit: ₹{pulp.value(prob_lp.objective):,.0f}\n")

# Collect results
lp_units   = {p: pulp.value(x[p])                   for p in PRODUCTS}
lp_revenue = {p: PROFIT[p]        * pulp.value(x[p]) for p in PRODUCTS}
lp_cost    = {p: VARIABLE_COST[p] * pulp.value(x[p]) for p in PRODUCTS}

lp_results_df = pd.DataFrame({
    "Product":     PRODUCTS,
    "Units":       [lp_units[p]                        for p in PRODUCTS],
    "Revenue (₹)": [f"{lp_revenue[p]:,.0f}"            for p in PRODUCTS],
    "Cost (₹)":    [f"{lp_cost[p]:,.0f}"               for p in PRODUCTS],
    "Profit (₹)":  [f"{lp_revenue[p]-lp_cost[p]:,.0f}" for p in PRODUCTS],
    "Demand Used": [f"{lp_units[p]/DEMAND[p]*100:.1f}%" for p in PRODUCTS],
}).set_index("Product")

print("📊 LP Optimal Production Plan:")
lp_results_df

In [ ]:
# ── Constraint Utilisation ──
machine_util = {
    m: sum(MACHINE_HOURS[p][m] * lp_units[p] for p in PRODUCTS)
    for m in MACHINES
}
material_util = {
    mat: sum(MATERIAL_USE[p][mat] * lp_units[p] for p in PRODUCTS)
    for mat in MATERIALS
}

util_df = pd.DataFrame([
    {"Resource": m, "Used": machine_util[m],
     "Capacity": MACHINE_CAPACITY[m],
     "Slack": MACHINE_CAPACITY[m] - machine_util[m],
     "Utilisation %": f"{machine_util[m]/MACHINE_CAPACITY[m]*100:.1f}%"}
    for m in MACHINES
] + [
    {"Resource": mat, "Used": material_util[mat],
     "Capacity": MATERIAL_SUPPLY[mat],
     "Slack": MATERIAL_SUPPLY[mat] - material_util[mat],
     "Utilisation %": f"{material_util[mat]/MATERIAL_SUPPLY[mat]*100:.1f}%"}
    for mat in MATERIALS
]).set_index("Resource")

total_cost = sum(lp_cost.values())
util_df.loc["Budget (₹)"] = [
    total_cost, WEEKLY_BUDGET, WEEKLY_BUDGET - total_cost,
    f"{total_cost/WEEKLY_BUDGET*100:.1f}%"
]

print("🔧 Resource Utilisation:")
util_df

---
## 5. Shadow Prices & Dual Analysis

**Shadow price** = how much the objective (profit) improves per unit increase in a constraint's RHS.  
- A shadow price of `₹30,000` on `Material_Semiconductors` means: **buy 1 more kg of semiconductors → earn ₹30,000 more profit**.  
- Zero shadow price = that constraint is not binding (there's slack).

In [ ]:
shadow_prices = {c.name: c.pi    for c in prob_lp.constraints.values()}
slack_values  = {c.name: c.slack for c in prob_lp.constraints.values()}

dual_df = pd.DataFrame({
    "Constraint":    list(shadow_prices.keys()),
    "Shadow Price (₹)": [f"{v:,.0f}" for v in shadow_prices.values()],
    "Slack":         [f"{v:.2f}"     for v in slack_values.values()],
    "Binding?": ["🔴 YES" if abs(v) < 0.01 else "🟢 No" for v in slack_values.values()],
}).set_index("Constraint")

print("📐 Dual / Shadow Price Analysis:")
dual_df

---
## 6. Model 2 — Integer Linear Programming (ILP)

Real products must be produced in whole numbers. ILP solves this with branch-and-bound.

In [ ]:
prob_ilp = pulp.LpProblem("Production_Planning_ILP", pulp.LpMaximize)

# Integer decision variables
xi = {p: pulp.LpVariable(f"xi_{p}", lowBound=0, cat="Integer") for p in PRODUCTS}

prob_ilp += pulp.lpSum(PROFIT[p] * xi[p] for p in PRODUCTS)

for p in PRODUCTS:
    prob_ilp += xi[p] <= DEMAND[p]
    prob_ilp += xi[p] >= MIN_PROD[p]

for m in MACHINES:
    prob_ilp += pulp.lpSum(MACHINE_HOURS[p][m] * xi[p] for p in PRODUCTS) <= MACHINE_CAPACITY[m]

for mat in MATERIALS:
    prob_ilp += pulp.lpSum(MATERIAL_USE[p][mat] * xi[p] for p in PRODUCTS) <= MATERIAL_SUPPLY[mat]

prob_ilp += pulp.lpSum(VARIABLE_COST[p] * xi[p] for p in PRODUCTS) <= WEEKLY_BUDGET

prob_ilp.solve(pulp.PULP_CBC_CMD(msg=0))

ilp_units  = {p: int(pulp.value(xi[p])) for p in PRODUCTS}
ilp_profit = pulp.value(prob_ilp.objective)

print(f"ILP Status : {pulp.LpStatus[prob_ilp.status]}")
print(f"ILP Profit : ₹{ilp_profit:,.0f}")
print(f"LP  Profit : ₹{pulp.value(prob_lp.objective):,.0f}")
print(f"ILP Gap    : ₹{pulp.value(prob_lp.objective) - ilp_profit:,.0f}")
print(f"\nILP Units  : {ilp_units}")

---
## 7. Sensitivity Analysis — Budget vs Profit

In [ ]:
budgets = list(range(3_000_000, 8_000_001, 500_000))
profits = []

for budget in budgets:
    p_sens = pulp.LpProblem("sens", pulp.LpMaximize)
    xs = {p: pulp.LpVariable(f"xs_{p}", lowBound=0, cat="Continuous") for p in PRODUCTS}
    p_sens += pulp.lpSum(PROFIT[p] * xs[p] for p in PRODUCTS)
    for p in PRODUCTS:
        p_sens += xs[p] <= DEMAND[p]
        p_sens += xs[p] >= MIN_PROD[p]
    for m in MACHINES:
        p_sens += pulp.lpSum(MACHINE_HOURS[p][m] * xs[p] for p in PRODUCTS) <= MACHINE_CAPACITY[m]
    for mat in MATERIALS:
        p_sens += pulp.lpSum(MATERIAL_USE[p][mat] * xs[p] for p in PRODUCTS) <= MATERIAL_SUPPLY[mat]
    p_sens += pulp.lpSum(VARIABLE_COST[p] * xs[p] for p in PRODUCTS) <= budget
    p_sens.solve(pulp.PULP_CBC_CMD(msg=0))
    profits.append(pulp.value(p_sens.objective) or 0)

sens_df = pd.DataFrame({
    "Budget (₹M)": [b/1e6 for b in budgets],
    "Max Profit (₹M)": [p/1e6 for p in profits],
    "Profit Margin %": [f"{(pr/b)*100:.1f}%" for pr, b in zip(profits, budgets)]
})

print("📈 Budget Sensitivity Results:")
sens_df

---
## 8. What-If Analysis — Semiconductor Supply Shock (−20%)

In [ ]:
shocked_supply = {**MATERIAL_SUPPLY, "Semiconductors": int(MATERIAL_SUPPLY["Semiconductors"] * 0.80)}
print(f"Normal semiconductor supply : {MATERIAL_SUPPLY['Semiconductors']} kg/week")
print(f"Shocked semiconductor supply: {shocked_supply['Semiconductors']} kg/week")

prob_shock = pulp.LpProblem("Supply_Shock", pulp.LpMaximize)
xs2 = {p: pulp.LpVariable(f"xs2_{p}", lowBound=0, cat="Continuous") for p in PRODUCTS}
prob_shock += pulp.lpSum(PROFIT[p] * xs2[p] for p in PRODUCTS)
for p in PRODUCTS:
    prob_shock += xs2[p] <= DEMAND[p]
    prob_shock += xs2[p] >= MIN_PROD[p]
for m in MACHINES:
    prob_shock += pulp.lpSum(MACHINE_HOURS[p][m] * xs2[p] for p in PRODUCTS) <= MACHINE_CAPACITY[m]
for mat in MATERIALS:
    prob_shock += pulp.lpSum(MATERIAL_USE[p][mat] * xs2[p] for p in PRODUCTS) <= shocked_supply[mat]
prob_shock += pulp.lpSum(VARIABLE_COST[p] * xs2[p] for p in PRODUCTS) <= WEEKLY_BUDGET

prob_shock.solve(pulp.PULP_CBC_CMD(msg=0))

shock_units  = {p: round(pulp.value(xs2[p]), 2) for p in PRODUCTS}
shock_profit = pulp.value(prob_shock.objective)

profit_drop = pulp.value(prob_lp.objective) - shock_profit

comparison_df = pd.DataFrame({
    "Product":        PRODUCTS,
    "Baseline Units": [lp_units[p]    for p in PRODUCTS],
    "Shock Units":    [shock_units[p] for p in PRODUCTS],
    "Change (units)": [shock_units[p] - lp_units[p] for p in PRODUCTS],
}).set_index("Product")

print(f"\n✅ Baseline Profit : ₹{pulp.value(prob_lp.objective):>12,.0f}")
print(f"⚠️  Shocked Profit  : ₹{shock_profit:>12,.0f}")
print(f"📉 Profit Drop     : ₹{profit_drop:>12,.0f} ({profit_drop/pulp.value(prob_lp.objective)*100:.1f}%)")
print("\nProduction changes:")
comparison_df

---
## 9. Visualisations

In [ ]:
COLORS = ["#3B82F6", "#10B981", "#F59E0B", "#EF4444", "#8B5CF6"]
STYLE_BG = "#F8FAFC"

# ── Figure 1: LP vs ILP Production Plan ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=STYLE_BG)

# Left: grouped bar — LP vs ILP
ax = axes[0]
ax.set_facecolor(STYLE_BG)
x_pos = np.arange(len(PRODUCTS))
w = 0.32
ax.bar(x_pos - w/2, [lp_units[p]  for p in PRODUCTS], w,
       label="LP (continuous)", color=COLORS[0], alpha=0.85)
ax.bar(x_pos + w/2, [ilp_units[p] for p in PRODUCTS], w,
       label="ILP (integer)",   color=COLORS[1], alpha=0.85)
ax.plot(x_pos, [DEMAND[p] for p in PRODUCTS], "D--", color="black",
        ms=6, alpha=0.5, label="Max Demand")
ax.set_xticks(x_pos); ax.set_xticklabels(PRODUCTS, fontsize=9)
ax.set_ylabel("Units / Week", fontsize=10)
ax.set_title("LP vs ILP — Optimal Production Plan", fontsize=11, fontweight="bold")
ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

# Right: profit breakdown by product
ax2 = axes[1]; ax2.set_facecolor(STYLE_BG)
profit_per = [(lp_revenue[p] - lp_cost[p])/1000 for p in PRODUCTS]
bars = ax2.bar(PRODUCTS, profit_per, color=COLORS, alpha=0.85, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, profit_per):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f"₹{val:.0f}K", ha="center", fontsize=9, fontweight="bold")
ax2.set_ylabel("Profit (₹ Thousands)", fontsize=10)
ax2.set_title("Weekly Profit Contribution per Product", fontsize=11, fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Figure 1 — Production Optimisation Results", fontsize=13,
             fontweight="bold", color="#1E293B", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Resource Utilisation ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor=STYLE_BG)

# Machine utilisation
ax = axes[0]; ax.set_facecolor(STYLE_BG)
m_pct = [machine_util[m] / MACHINE_CAPACITY[m] * 100 for m in MACHINES]
bar_cols = [COLORS[3] if v > 90 else COLORS[0] for v in m_pct]
bars = ax.barh(MACHINES, m_pct, color=bar_cols, alpha=0.85)
ax.axvline(100, color="red", lw=1.5, ls="--", label="100% capacity")
ax.axvline(80,  color="orange", lw=1.2, ls=":",  label="80% warning")
for bar, val in zip(bars, m_pct):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=11, fontweight="bold")
ax.set_xlim(0, 115); ax.set_xlabel("Utilisation (%)", fontsize=10)
ax.set_title("Machine Utilisation", fontsize=11, fontweight="bold")
ax.legend(fontsize=8); ax.grid(axis="x", alpha=0.3)

# Material utilisation
ax2 = axes[1]; ax2.set_facecolor(STYLE_BG)
mat_pct = [material_util[mat] / MATERIAL_SUPPLY[mat] * 100 for mat in MATERIALS]
bar_cols2 = [COLORS[3] if v > 95 else COLORS[1] for v in mat_pct]
bars2 = ax2.barh(MATERIALS, mat_pct, color=bar_cols2, alpha=0.85)
ax2.axvline(100, color="red", lw=1.5, ls="--")
for bar, val in zip(bars2, mat_pct):
    ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}%", va="center", fontsize=11, fontweight="bold")
ax2.set_xlim(0, 115); ax2.set_xlabel("Utilisation (%)", fontsize=10)
ax2.set_title("Raw Material Utilisation", fontsize=11, fontweight="bold")
ax2.grid(axis="x", alpha=0.3)

plt.suptitle("Figure 2 — Constraint Utilisation Analysis", fontsize=13,
             fontweight="bold", color="#1E293B", y=1.02)
plt.tight_layout()
plt.show()

print("💡 Insight: Semiconductors are the BINDING constraint (100% used).")
print("   Machines and budget have slack — buying more semiconductors gives the most uplift.")

In [ ]:
# ── Figure 3: Sensitivity Analysis ───────────────────────────
fig, ax = plt.subplots(figsize=(11, 4.5), facecolor=STYLE_BG)
ax.set_facecolor(STYLE_BG)

b_m = [b/1e6 for b in budgets]
p_m = [p/1e6 for p in profits]

ax.plot(b_m, p_m, "o-", color=COLORS[0], lw=2.5, ms=8)
ax.fill_between(b_m, p_m, alpha=0.12, color=COLORS[0])
ax.axvline(WEEKLY_BUDGET/1e6, color=COLORS[3], ls="--", lw=2,
           label=f"Current Budget = ₹{WEEKLY_BUDGET/1e6:.1f}M")
ax.axhline(pulp.value(prob_lp.objective)/1e6, color=COLORS[1], ls=":", lw=1.5,
           label=f"Current Optimal Profit = ₹{pulp.value(prob_lp.objective)/1e6:.2f}M")

# Annotate points
for b, p in zip(b_m, p_m):
    ax.annotate(f"₹{p:.2f}M", (b, p), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=8, color="#1E293B")

ax.set_xlabel("Weekly Budget (₹ Millions)", fontsize=11)
ax.set_ylabel("Maximum Achievable Profit (₹ Millions)", fontsize=11)
ax.set_title("Sensitivity Analysis — Optimal Profit vs Weekly Budget",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 4: Supply Shock Impact ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor=STYLE_BG)

# Grouped bar
ax = axes[0]; ax.set_facecolor(STYLE_BG)
x_pos = np.arange(len(PRODUCTS))
w = 0.32
ax.bar(x_pos - w/2, [lp_units[p]    for p in PRODUCTS], w,
       color=COLORS[0], alpha=0.85, label="Baseline")
ax.bar(x_pos + w/2, [shock_units[p] for p in PRODUCTS], w,
       color=COLORS[3], alpha=0.85, label="Chip Shortage −20%")
ax.set_xticks(x_pos); ax.set_xticklabels(PRODUCTS, fontsize=9)
ax.set_ylabel("Units / Week"); ax.legend(fontsize=9)
ax.set_title("Production Plan — Baseline vs Shock", fontsize=11, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# KPI comparison
ax2 = axes[1]; ax2.axis("off")
lp_p = pulp.value(prob_lp.objective)
table_data = [
    ["Metric", "Baseline", "After Shock", "Change"],
    ["Weekly Profit", f"₹{lp_p/1e6:.2f}M", f"₹{shock_profit/1e6:.2f}M",
     f"▼ ₹{profit_drop/1e6:.2f}M"],
    ["Profit Impact %", "100%", f"{shock_profit/lp_p*100:.1f}%",
     f"−{profit_drop/lp_p*100:.1f}%"],
    ["Semiconductor Supply", "300 kg", "240 kg", "−60 kg"],
]
tbl = ax2.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc="center", loc="center", colWidths=[0.3, 0.22, 0.25, 0.23])
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.1, 2.0)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#1E293B"); cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#EFF6FF")
ax2.set_title("KPI Impact Summary", fontsize=11, fontweight="bold", pad=16)

plt.suptitle("Figure 4 — What-If Analysis: Semiconductor Supply Shock",
             fontsize=13, fontweight="bold", color="#1E293B", y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Key Insights & Business Recommendations

### 🔍 What the Optimisation Found

| Finding | Detail |
|---|---|
| **Maximum weekly profit** | ₹90,77,500 |
| **Binding constraint** | Semiconductor supply (100% used) |
| **Non-binding constraints** | All machines, budget, plastics, metals |
| **Most profitable product** | Laptop (₹15K/unit profit) — limited by semiconductors |
| **Volume champion** | Smartphone + Headphones — fully utilise semiconductor allocation |

### 💡 Business Recommendations

1. **Invest in semiconductor supply.** Shadow price = ₹30,000/kg. Every additional kg of semiconductor supply translates to ₹30,000 more weekly profit. This is the single highest-ROI investment available to management.

2. **Budget is not the bottleneck.** Only 63.8% of the weekly budget is used. Simply increasing budget will NOT improve profit — the semiconductor bottleneck must be resolved first.

3. **Tablet & Smartwatch production is at contractual minimums.** These products are net drains on semiconductor capacity relative to their profit margins compared to Smartphones. Consider renegotiating minimum contracts.

4. **Semiconductor supply shock costs ₹18 lakh/week.** Supply chain resilience for chips should be a strategic priority — dual-sourcing or buffer stock is justified.

5. **Machines have headroom.** Assembly (87.7%), Testing (84.1%), Packaging (64.0%) have capacity. If semiconductor supply increases, production can scale up without new machine investment.

---
## 11. Comprehensive Dashboard

In [ ]:
# Run solver.py to regenerate the full dashboard PNG
import subprocess, os
result = subprocess.run(["python3", "solver.py"], capture_output=True, text=True)
print(result.stdout)

# Display it
from IPython.display import Image
Image("optimization_dashboard.png", width=950)

---
## 12. Summary

```
╔══════════════════════════════════════════════════════╗
║     PRODUCTION PLANNING OPTIMISATION — SUMMARY      ║
╠══════════════════════════════════════════════════════╣
║  Model            : Linear Programming (PuLP/CBC)   ║
║  Decision Vars    : 5 (units per product)            ║
║  Constraints      : 16                               ║
╠══════════════════════════════════════════════════════╣
║  LP  Optimal Profit : ₹90,77,500 / week             ║
║  ILP Optimal Profit : ₹90,77,500 / week             ║
║  LP–ILP Gap         : ₹0 (perfectly integer sol.)   ║
╠══════════════════════════════════════════════════════╣
║  Binding Constraint : Semiconductor Supply           ║
║  Shadow Price       : ₹30,000 / kg of chips         ║
║  Supply Shock Impact: −₹18,00,000 / week (−20% chip)║
╚══════════════════════════════════════════════════════╝
```

---
*Kartikay Verma | B.Tech CSE (Data Science) | Amity University Noida*